# EIIE 베이스라인 — 최신 RL 트레이딩 논문 비교 (`paper/GAP_ANALYSIS.md` 신규 #3)

지금까지 베이스라인 비교(`crypto_baselines_colab.ipynb`)는 등가중·Buy&Hold·모멘텀·MVO 같은
**고전 전략**뿐이었다. EarnHFT/FreQuant/DeepClair 등 이 프로젝트가 참고한 논문들은 실제로
**EIIE**(Jiang, Xu, & Liang, 2017, *A Deep Reinforcement Learning Framework for the Financial
Portfolio Management Problem*, arXiv:1706.10059)를 표준 비교 대상으로 쓴다 — CNN 기반 정책 +
Portfolio-Vector Memory(PVM, 직전 비중을 다음 결정의 입력으로 재사용) + 현금 편향(cash bias)이
핵심 아이디어이며, **원 논문이 애초에 크립토 포트폴리오에 적용된 논문**이라 이 프로젝트와 도메인이
정확히 일치한다.

**이식 범위(간소화 고지)**: EIIE의 네트워크 토폴로지(자산별 독립·가중치공유 CNN 평가기 +
PVM + 현금 편향 → softmax 비중)는 그대로 구현하지만, 원 논문의 학습 알고리즘(Online Stochastic
Batch Learning, 기하분포 표본추출 + 미분가능한 백테스트 직접 최적화)은 재현하지 않고 **이
프로젝트의 기존 PPO 파이프라인으로 대체**한다 — 네트워크 구조를 정책망의 feature extractor로
꽂아넣고 PPO로 학습시키는 방식. 우리 방법(Chronos+MTGNN 임베딩)의 정규화 트릭(우선순위 청크
샘플링 등)은 EIIE에 안 씀 — 순수 "바깥 베이스라인"으로 유지하기 위함.

**공정 비교를 위한 설계**: 배포 모델(β=-30)과 **동일한 train(2025 전체)/valid(2025 Q4)/test(2026 H1)
버킷, 동일 거래비용(TC=0.001), 동일 학습 예산(100k 스텝)**으로 맞춘다. 다만 상태 표현은 EIIE
고유의 원시 가격 윈도우(Chronos+MTGNN 임베딩 아님)를 쓴다 — `crypto_rl_embeddings.h5`에 의존하지
않고 `features_5m` 원시 OHLC만으로 자기완결적으로 동작.


In [1]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q stable-baselines3[extra] gymnasium pyarrow


Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 93.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.6/187.6 kB 12.5 MB/s eta 0:00:00


## 1. 설정

In [2]:
import warnings
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

import torch
import torch.nn as nn
import gymnasium as gym
from gymnasium import spaces
from stable_baselines3 import PPO
from stable_baselines3.common.env_checker import check_env
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor

warnings.filterwarnings('ignore')
plt.rcParams['axes.unicode_minus'] = False

DRIVE_ROOT = Path('/content/drive/MyDrive/졸업프로젝트')
DATA_DIR   = DRIVE_ROOT / 'data'
CRYPTO_DIR = DATA_DIR / 'crypto'
FEAT_DIR   = CRYPTO_DIR / 'features_5m'
MODEL_DIR  = DRIVE_ROOT / 'models'

TC         = 0.001
BUCKET_SEC = 1800

RL_TRAIN_START = '2025-01-01'
RL_TRAIN_END   = '2025-12-31'
RL_TEST_START  = '2026-01-01'
RL_TEST_END    = '2099-12-31'
VALID_START, VALID_END = '2025-10-01', '2025-12-31'

WINDOW       = 50    # EIIE 원 논문 기본값
N_CHANNELS   = 3     # [close, high, low] / 마지막 종가 비율
EIIE_TIMESTEPS = 100_000   # β=-30(§13-C)와 동일 예산

EIIE_MODEL_DIR = MODEL_DIR / 'crypto_eiie_baseline'
EIIE_MODEL_DIR.mkdir(parents=True, exist_ok=True)

print('경로 확인 완료')


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


경로 확인 완료


## 2. 데이터 로드 — 심볼 목록 + RetTarget_6b (임베딩 불필요, features_5m만 사용)

In [3]:
files = sorted(glob.glob(str(FEAT_DIR / '*.parquet')))
assert files, f'피처 parquet 없음: {FEAT_DIR}'
# [2026-10-04] 2026-08-18 PIT walk-forward용으로 추가된 3종목(backfill_pit_extra.py)은 원 44종목 유니버스가 아님 — 제외
PIT_EXTRA = {'DOTUSDT', 'THETAUSDT', 'XTZUSDT'}
files = [fp for fp in files if Path(fp).stem not in PIT_EXTRA]
symbols = sorted(Path(fp).stem for fp in files)
N_ASSETS = len(symbols)
assert N_ASSETS == 44, f'유니버스가 44종목이 아님({N_ASSETS}) — 저장된 모델과 차원 불일치'
print(f'심볼 {N_ASSETS}개: {symbols[:5]} ...')

rl_start_ts = pd.Timestamp(RL_TRAIN_START, tz='UTC')
ret_index = {}
all_buckets_set = set()

for fp in files:
    sym = Path(fp).stem
    df = pd.read_parquet(fp, columns=['datetime', 'RetTarget_6b'])
    df = df[df['datetime'] >= rl_start_ts]
    epoch = df['datetime'].values.astype('int64') // 10**9
    y     = df['RetTarget_6b'].values
    b     = epoch // BUCKET_SEC
    for bb, r in zip(b, y):
        if not np.isnan(r):
            bb = int(bb)
            ret_index[(sym, bb)] = float(r)
            all_buckets_set.add(bb)

all_buckets = sorted(all_buckets_set)
print(f'수익률 인덱스: {len(ret_index):,}개 | 전체 30분 버킷: {len(all_buckets):,}개')


심볼 44개: ['AAVEUSDT', 'ADAUSDT', 'ALLOUSDT', 'ARBUSDT', 'AVAXUSDT'] ...
수익률 인덱스: 1,079,569개 | 전체 30분 버킷: 26,841개


## 3. 버킷 분할

In [4]:
def bucket_to_ts(b):
    return pd.Timestamp(b * BUCKET_SEC, unit='s', tz='UTC')

def bucket_to_date(b):
    return bucket_to_ts(b).strftime('%Y-%m-%d')

train_buckets = [b for b in all_buckets if RL_TRAIN_START <= bucket_to_date(b) <= RL_TRAIN_END]
valid_buckets = [b for b in train_buckets if VALID_START <= bucket_to_date(b) <= VALID_END]
test_buckets  = [b for b in all_buckets if RL_TEST_START  <= bucket_to_date(b) <= RL_TEST_END]

print(f'train: {bucket_to_ts(train_buckets[0])} ~ {bucket_to_ts(train_buckets[-1])} ({len(train_buckets):,}스텝)')
print(f'valid: {bucket_to_ts(valid_buckets[0])} ~ {bucket_to_ts(valid_buckets[-1])} ({len(valid_buckets):,}스텝)')
print(f'test : {bucket_to_ts(test_buckets[0])} ~ {bucket_to_ts(test_buckets[-1])} ({len(test_buckets):,}스텝)')


train: 2025-01-01 00:00:00+00:00 ~ 2025-12-31 23:30:00+00:00 (17,520스텝)
valid: 2025-10-01 00:00:00+00:00 ~ 2025-12-31 23:30:00+00:00 (4,416스텝)
test : 2026-01-01 00:00:00+00:00 ~ 2026-07-14 04:00:00+00:00 (9,321스텝)


## 4. 원시 가격 행렬 구성 — EIIE 입력용 (close/high/low, 30분 리샘플)

In [5]:
%%time
BUCKET_MIN = train_buckets[0] - WINDOW - 5   # 워밍업 여유
BUCKET_MAX = test_buckets[-1]
global_idx = np.arange(BUCKET_MIN, BUCKET_MAX + 1)
pos_of = {b: i for i, b in enumerate(global_idx)}

close_mat = np.full((N_ASSETS, len(global_idx)), np.nan, dtype=np.float32)
high_mat  = np.full_like(close_mat, np.nan)
low_mat   = np.full_like(close_mat, np.nan)

for i, sym in enumerate(symbols):
    df = pd.read_parquet(FEAT_DIR / f'{sym}.parquet', columns=['datetime', 'open', 'high', 'low', 'close'])
    epoch = df['datetime'].values.astype('int64') // 10**9
    b = epoch // BUCKET_SEC
    tmp = pd.DataFrame({'high': df['high'].values, 'low': df['low'].values, 'close': df['close'].values}, index=b)
    agg = tmp.groupby(level=0).agg({'high': 'max', 'low': 'min', 'close': 'last'})
    agg = agg.reindex(global_idx)
    agg = agg.ffill().bfill()   # 중간 결측=직전값 유지, 상장 전(전체 구간 이전)=최초 상장가로 backfill(중립 취급)
    close_mat[i] = agg['close'].to_numpy(dtype=np.float32)
    high_mat[i]  = agg['high'].to_numpy(dtype=np.float32)
    low_mat[i]   = agg['low'].to_numpy(dtype=np.float32)

assert not np.isnan(close_mat).any(), '가격 행렬에 NaN 남음 — 전 구간 데이터 없는 심볼 확인 필요'
print(f'가격 행렬: {close_mat.shape} (심볼 × 버킷)')


가격 행렬: (44, 26896) (심볼 × 버킷)
CPU times: user 4.05 s, sys: 887 ms, total: 4.94 s
Wall time: 28.5 s


## 5. EIIEPortfolioEnv — 원시 가격 윈도우 + PVM + 명시적 현금

In [6]:
# [2026-10-04 회계 수정] 가격 변동 후 실제 보유 비중(drift)·거래 가능 종목 마스크.
# 이전 버전은 수익 반영 뒤에도 직전 '목표' 비중을 보유 비중으로 저장해, 가격 변동으로 틀어진 비중을
# 되돌리는 재조정 거래(등가중 포함)의 비용이 0으로 계산됐다.
def _drift_weights(w, rets):
    """수익 반영 뒤 실제 보유 비중 w_i(1+r_i)/(1+wᵀr). 합<1이면 나머지는 현금(수익률 0)으로 남는다."""
    w = np.asarray(w, dtype=np.float64)
    g = 1.0 + float(np.dot(w, rets))
    if g <= 1e-12:
        return w.astype(np.float32)
    return (w * (1.0 + np.asarray(rets, dtype=np.float64)) / g).astype(np.float32)


def _mask_untradable(w, tradable, renorm=True):
    """수익률 데이터가 없는(상장 전·상폐 후) 종목 비중을 0으로. renorm=False면 빠진 몫은 현금으로 남는다."""
    w = np.where(tradable, np.asarray(w, dtype=np.float64), 0.0)
    s = w.sum()
    return (w / s if renorm and s > 1e-12 else w).astype(np.float32)


class EIIEPortfolioEnv(gym.Env):
    """EIIE(Jiang et al. 2017) 상태 표현 이식 — 원시 가격 윈도우 + 직전 비중(PVM).
    Action: (N+1,)-dim 원시 점수(자산 N개 + 현금 1개) → softmax로 비중 변환.
    거래비용은 자산 레그에만 부과(v3, crypto_hrl_earnhft_colab.ipynb §12-A와 동일 관례)."""
    metadata = {'render_modes': []}

    def __init__(self, buckets, symbols, close_mat, high_mat, low_mat, pos_of,
                 window=WINDOW, tc=TC):
        super().__init__()
        self.buckets   = buckets
        self.symbols   = symbols
        self.n_assets  = len(symbols)
        self.close_mat = close_mat
        self.high_mat  = high_mat
        self.low_mat   = low_mat
        self.pos_of    = pos_of
        self.window    = window
        self.tc        = tc

        obs_dim = self.n_assets * N_CHANNELS * window + self.n_assets
        self.observation_space = spaces.Box(-np.inf, np.inf, shape=(obs_dim,), dtype=np.float32)
        self.action_space      = spaces.Box(-3.0, 3.0, shape=(self.n_assets + 1,), dtype=np.float32)

    def _price_window(self, bucket):
        pos = self.pos_of[bucket]
        sl = slice(pos - self.window + 1, pos + 1)
        c = self.close_mat[:, sl]
        h = self.high_mat[:, sl]
        l = self.low_mat[:, sl]
        last_close = c[:, -1:]
        last_close = np.where(last_close == 0, 1e-8, last_close)
        rc = (c / last_close).astype(np.float32)
        rh = (h / last_close).astype(np.float32)
        rl = (l / last_close).astype(np.float32)
        return np.stack([rc, rh, rl], axis=1)   # (n_assets, 3, window)

    def _asset_ret(self, bucket):
        rets = np.array([ret_index.get((sym, bucket), 0.0) for sym in self.symbols], dtype=np.float32)
        return np.concatenate([rets, [0.0]])   # 마지막 = 현금(수익률 0)

    def _tradable(self, bucket):
        """거래 가능 마스크(마지막 = 현금, 항상 가능)"""
        return np.array([(sym, bucket) in ret_index for sym in self.symbols] + [True])

    def _get_obs(self):
        b = self.buckets[self.t]
        pw = self._price_window(b)
        return np.concatenate([pw.flatten(), self.weights[:-1]]).astype(np.float32)

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.t               = 0
        self.weights         = np.zeros(self.n_assets + 1, dtype=np.float32)
        self.weights[-1]     = 1.0   # 시작은 전액 현금
        self.portfolio_value = 1.0
        self.value_history   = [1.0]
        self.turnover_history = []
        return self._get_obs(), {}

    def step(self, action):
        action = np.exp(action - action.max())
        w = (action / action.sum()).astype(np.float32)
        w = _mask_untradable(w, self._tradable(self.buckets[self.t]))  # 거래 불가 종목 제외

        turnover = float(np.abs(w[:-1] - self.weights[:-1]).sum())   # 현금 제외(자산 레그만 비용)
        tc_cost  = self.tc * turnover

        b = self.buckets[self.t]
        asset_rets = self._asset_ret(b)
        port_ret = float((w * asset_rets).sum())

        self.portfolio_value *= (1 + port_ret - tc_cost)
        self.weights = _drift_weights(w, asset_rets)  # 가격 변동 후 실제 보유 비중
        self.value_history.append(self.portfolio_value)
        self.turnover_history.append(turnover)
        self.t += 1

        done = (self.t >= len(self.buckets) - 1)
        reward = float(port_ret - tc_cost)
        info = {'portfolio_value': self.portfolio_value, 'port_ret': port_ret, 'turnover': turnover}
        return self._get_obs(), reward, done, False, info


# 스모크 테스트
_env = EIIEPortfolioEnv(train_buckets[:200], symbols, close_mat, high_mat, low_mat, pos_of)
check_env(_env, warn=True)
obs, _ = _env.reset()
a = np.random.randn(_env.n_assets + 1).astype(np.float32)
obs, r, done, _, info = _env.step(a)
print(f'obs shape: {obs.shape} | action shape: {_env.action_space.shape} | reward: {r:.5f}')


obs shape: (6644,) | action shape: (45,) | reward: 0.00465


## 6. EIIE 네트워크 (CNN 평가기 shared weights + PVM + 현금 편향)

In [7]:
class EIIETorchNet(nn.Module):
    """Jiang et al. (2017) Fig.2 토폴로지 이식 — 자산별 독립(가중치 공유) CNN 평가기.
    conv1(kernel=3) → conv2(kernel=window-2, 남은 시간축을 1로 축소) → PVM(직전 비중) 결합
    → 자산별 스칼라 점수 → 학습 가능한 현금 편향과 concat."""

    def __init__(self, n_assets, window, n_channels=N_CHANNELS, conv1_out=2, conv2_out=20):
        super().__init__()
        self.n_assets, self.window, self.n_channels = n_assets, window, n_channels
        self.conv1 = nn.Conv1d(n_channels, conv1_out, kernel_size=3)
        conv1_len = window - 3 + 1
        self.conv2 = nn.Conv1d(conv1_out, conv2_out, kernel_size=conv1_len)
        self.score_head = nn.Linear(conv2_out + 1, 1)   # +1 = PVM(직전 비중)
        self.cash_bias = nn.Parameter(torch.zeros(1))

    def forward(self, price_window, prev_weights):
        # price_window: (B, n_assets, n_channels, window), prev_weights: (B, n_assets)
        B = price_window.shape[0]
        x = price_window.reshape(B * self.n_assets, self.n_channels, self.window)
        x = torch.relu(self.conv1(x))
        x = torch.relu(self.conv2(x))
        x = x.reshape(B, self.n_assets, -1)
        pw = prev_weights.unsqueeze(-1)
        x = torch.cat([x, pw], dim=-1)
        scores = self.score_head(x).squeeze(-1)          # (B, n_assets)
        cash = self.cash_bias.expand(B, 1)
        return torch.cat([scores, cash], dim=-1)          # (B, n_assets+1)


class EIIEFeaturesExtractor(BaseFeaturesExtractor):
    """SB3 정책망에 꽂는 래퍼 — 관측(flatten)을 다시 (가격윈도우, 직전비중)으로 분리해 EIIETorchNet에 전달."""

    def __init__(self, observation_space, n_assets=N_ASSETS, window=WINDOW, n_channels=N_CHANNELS):
        features_dim = n_assets + 1
        super().__init__(observation_space, features_dim=features_dim)
        self.n_assets, self.window, self.n_channels = n_assets, window, n_channels
        self.net = EIIETorchNet(n_assets, window, n_channels)
        self._pw_size = n_assets * n_channels * window

    def forward(self, observations):
        pw_flat = observations[:, :self._pw_size]
        prev_w  = observations[:, self._pw_size:]
        pw = pw_flat.reshape(-1, self.n_assets, self.n_channels, self.window)
        return self.net(pw, prev_w)


policy_kwargs = dict(
    features_extractor_class=EIIEFeaturesExtractor,
    features_extractor_kwargs=dict(n_assets=N_ASSETS, window=WINDOW, n_channels=N_CHANNELS),
    net_arch=[],   # feature extractor 출력(자산별 점수+현금 편향)을 거의 그대로 정책에 사용(EIIE 토폴로지 보존)
)
print('EIIE 네트워크 정의 완료')


EIIE 네트워크 정의 완료


## 7. 성과 계산 함수

In [8]:
def compute_perf(values, steps_per_year=48 * 365):
    rets       = np.diff(values) / values[:-1]
    total_ret  = values[-1] / values[0] - 1
    annual_ret = (1 + total_ret) ** (steps_per_year / len(rets)) - 1
    sharpe     = rets.mean() / (rets.std() + 1e-8) * np.sqrt(steps_per_year)
    max_dd     = ((values / np.maximum.accumulate(values)) - 1).min()
    return {'총 수익률': total_ret, '연환산 수익률': annual_ret, 'Sharpe': sharpe, 'MDD': max_dd}


def perf_row(values, turnovers, tc=TC):
    perf = compute_perf(values)
    t = np.asarray(turnovers, dtype=np.float64)
    perf['평균 턴오버/스텝'] = t.mean()
    perf['누적 비용 드래그'] = 1 - np.prod(1 - tc * t)
    return perf


def equal_weight_backtest_eiie(buckets):
    env = EIIEPortfolioEnv(buckets, symbols, close_mat, high_mat, low_mat, pos_of)
    obs, _ = env.reset()
    done = False
    action = np.zeros(env.n_assets + 1, dtype=np.float32)
    action[:-1] = 1.0   # 현금 제외 자산에 동일 로짓 → softmax 후 자산 균등, 현금 0
    action[-1] = -10.0  # 현금 로짓을 낮게 줘서 현금 비중이 거의 0이 되도록
    while not done:
        obs, _, done, _, info = env.step(action)
    return np.array(env.value_history), list(env.turnover_history)


## 8. EIIE 학습 (PPO, β=-30과 동일 예산 100k 스텝, 단일 연속 에피소드 — 우리 방법의 청크 샘플링 안 씀)

In [9]:
%%time
train_env = Monitor(EIIEPortfolioEnv(train_buckets, symbols, close_mat, high_mat, low_mat, pos_of))

eiie_model = PPO(
    'MlpPolicy', train_env,
    learning_rate=3e-4, n_steps=1024, batch_size=64, n_epochs=10,
    gamma=0.99, gae_lambda=0.95, clip_range=0.2, ent_coef=0.01, vf_coef=0.5,
    max_grad_norm=0.5, policy_kwargs=policy_kwargs,
    verbose=0, device='cpu', seed=42,
)
eiie_model.learn(total_timesteps=EIIE_TIMESTEPS, progress_bar=True)

eiie_model.save(str(EIIE_MODEL_DIR / 'eiie_seed42.zip'))
print('저장 완료: eiie_seed42.zip')


Output()

저장 완료: eiie_seed42.zip
CPU times: user 9min 50s, sys: 4.5 s, total: 9min 55s
Wall time: 10min 9s


In [10]:
# 재실행용 로더 (idempotent)
if 'eiie_model' not in globals():
    eiie_model = PPO.load(str(EIIE_MODEL_DIR / 'eiie_seed42.zip'),
                           custom_objects={'policy_kwargs': policy_kwargs})
    print('EIIE 모델 로드 완료')


## 9. 평가 — valid/test, 기존 결과와 비교

In [11]:
def run_eiie(model, buckets):
    env = EIIEPortfolioEnv(buckets, symbols, close_mat, high_mat, low_mat, pos_of)
    obs, _ = env.reset()
    done = False
    while not done:
        a, _ = model.predict(obs, deterministic=True)
        obs, _, done, _, info = env.step(a)
    return np.array(env.value_history), list(env.turnover_history)


print('=== EIIE 평가 — valid vs test ===\n')

ew_valid_vals, ew_valid_turns = equal_weight_backtest_eiie(valid_buckets)
ew_test_vals,  ew_test_turns  = equal_weight_backtest_eiie(test_buckets)

eiie_valid_vals, eiie_valid_turns = run_eiie(eiie_model, valid_buckets)
eiie_test_vals,  eiie_test_turns  = run_eiie(eiie_model, test_buckets)

comparison_valid = {
    '등가중 벤치마크(이 노트북 재계산)': perf_row(ew_valid_vals, ew_valid_turns),
    'EIIE(PPO 학습, seed=42)':          perf_row(eiie_valid_vals, eiie_valid_turns),
}
comparison_test = {
    '등가중 벤치마크(이 노트북 재계산)': perf_row(ew_test_vals, ew_test_turns),
    'EIIE(PPO 학습, seed=42)':          perf_row(eiie_test_vals, eiie_test_turns),
    # 참고용 — 다른 노트북에서 이미 확인된 값(동일 버킷·비용 기준, 재계산 아님)
    'β=-30 RL(배포, 참고치)':           {'총 수익률': 0.0566, 'Sharpe': 1.1556, 'MDD': -0.0743},
    '모멘텀 top-10(참고치)':            {'총 수익률': 1.0604, 'Sharpe': 2.4433, 'MDD': -0.2796},
    'MVO(Ledoit-Wolf, 참고치)':         {'총 수익률': -0.0910, 'Sharpe': -0.1294, 'MDD': -0.4009},
}

valid_df = pd.DataFrame(comparison_valid).T
test_df  = pd.DataFrame(comparison_test).T

print('--- valid(2025 Q4) ---')
print(valid_df.to_string(float_format=lambda x: f'{x:.4f}'))
print('\n--- test(2026~) ---')
print(test_df.to_string(float_format=lambda x: f'{x:.4f}'))

valid_df.to_csv(CRYPTO_DIR / 'crypto_eiie_valid.csv', encoding='utf-8-sig')
test_df.to_csv(CRYPTO_DIR / 'crypto_eiie_test.csv', encoding='utf-8-sig')
print('\n저장 완료: crypto_eiie_valid.csv, crypto_eiie_test.csv')
print('\n(주의: 참고치 3개 행은 이 노트북에서 재계산한 게 아니라 다른 노트북 결과를 그대로 옮겨온 것 —')
print(' 버킷·TC는 동일하게 맞췄으나 완전한 재현은 아님. 엄밀 비교는 등가중·EIIE 두 행 사이에서만 유효.)')


=== EIIE 평가 — valid vs test ===

--- valid(2025 Q4) ---
                        총 수익률  연환산 수익률  Sharpe     MDD  평균 턴오버/스텝  누적 비용 드래그
등가중 벤치마크(이 노트북 재계산)   -0.3332  -0.7998 -0.9815 -0.4425     0.0002     0.0010
EIIE(PPO 학습, seed=42) -0.3159  -0.7783 -0.9686 -0.4310     0.0010     0.0045

--- test(2026~) ---
                        총 수익률  연환산 수익률  Sharpe     MDD  평균 턴오버/스텝  누적 비용 드래그
등가중 벤치마크(이 노트북 재계산)    0.0159   0.0301  0.3296 -0.3959     0.0001     0.0010
EIIE(PPO 학습, seed=42)  0.0018   0.0033  0.2751 -0.3882     0.0007     0.0063
β=-30 RL(배포, 참고치)      0.0566      NaN  1.1556 -0.0743        NaN        NaN
모멘텀 top-10(참고치)        1.0604      NaN  2.4433 -0.2796        NaN        NaN
MVO(Ledoit-Wolf, 참고치) -0.0910      NaN -0.1294 -0.4009        NaN        NaN

저장 완료: crypto_eiie_valid.csv, crypto_eiie_test.csv

(주의: 참고치 3개 행은 이 노트북에서 재계산한 게 아니라 다른 노트북 결과를 그대로 옮겨온 것 —
 버킷·TC는 동일하게 맞췄으나 완전한 재현은 아님. 엄밀 비교는 등가중·EIIE 두 행 사이에서만 유효.)


## 10. 판정 기준

- **EIIE가 등가중은 이기지만 β=-30 RL(참고치)에는 못 미친다** → 우리 방법(Chronos+MTGNN 임베딩 +
  top-K 희소선택 + 관성 스무딩)이 원시 가격만 쓰는 표준 RL 베이스라인 대비 실질적 가치를 더한다는
  근거 — 논문의 "왜 우리 방법이 필요한가" 핵심 증거로 사용.
- **EIIE가 β=-30과 비슷하거나 더 낫다** → 임베딩·희소선택 같은 이 프로젝트의 추가 장치들이
  단순 CNN 대비 유의미한 이득을 못 준다는 뜻 — 정직하게 그렇게 보고하고, 어느 구성요소가 실제로
  기여하는지 재검토 필요.
- **EIIE도 valid→test 낙차가 크면**(모멘텀처럼) → "raw 가격만 쓰는 모델은 이 시장에서 일관성이
  떨어진다"는 이 프로젝트의 반복된 발견(§16, `ABLATION_TABLE.md` 표2)이 세 번째 사례로 재확인되는 것.

결과를 `paper/ABLATION_TABLE.md` 표 2에 EIIE 행으로 추가하고 `paper/FINDINGS_LOG.md`에 반영할 것.

⚠️ **한계**: OSBL(기하분포 배치 샘플링)·직접 미분가능 백테스트 최적화라는 원 논문 학습법 대신
PPO를 썼다는 간소화가 있음 — "EIIE 네트워크 토폴로지 + 우리 PPO 파이프라인"이지 "EIIE 논문의
완전한 재현"은 아니라는 점을 논문에 명시할 것.
